---
title: Funding & open access
description: Funding acknowledgements, the most frequent funders and funder groups, EU funding and open access of the publications of the School of Electrical and Computer Engineering, Aristotle University of Thessaloniki, in Scopus.
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "sece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sece_stats import config, data
from sece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the Scopus export, partial
AS_OF = config.EXPORT_DATE      # the export date (datetime.date)

In [ ]:
import re
import unicodedata

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sece_stats import periods

pubs = data.load_publications()
funders = data.load_publication_funders()

FIRST_YEAR = int(pubs["year"].min())
LAST_FULL = periods.LAST_FULL                # latest full year: the year before PARTIAL
AS_OF_TEXT = f"{AS_OF.day} {AS_OF:%B %Y}"    # the export date in words (day, month, year)
KIND_COLORS = dp.CATEGORY_COLORS["source_kind"]

# Five-year periods that end with the latest full year (sece_stats.periods; the first one is clipped to the data);
# the partial year has a label of its own and stays out of every period.
pubs["period"] = pubs["year"].map(lambda year: periods.period_of(year, first_year=FIRST_YEAR))
P_RECENT = periods.period_of(LAST_FULL)       # the latest five full years
P_PREV = periods.period_of(LAST_FULL - 5)     # the five years before them
P_DECADE = periods.period_of(LAST_FULL - 10)  # ten years before P_RECENT
RECENT_FROM = periods.bounds(P_RECENT)[0]     # first year of P_RECENT
SPLIT = periods.bounds(P_PREV)[0]             # first year of the later periods (P_PREV and P_RECENT)


def and_join(items) -> str:
    """'a', 'a and b', 'a, b and c'."""
    items = [str(item) for item in items]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


def wilson(k, n, z: float = 1.96):
    """95% Wilson score interval of a proportion k/n (arrays allowed)."""
    k, n = np.asarray(k, dtype=float), np.asarray(n, dtype=float)
    p = k / n
    den = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return centre - half, centre + half


# --------------------------------------------------------------------------- funder normalisation
# Scopus "Funding Details" are free text. Ordered rules on a folded key (no accents, lower case;
# first match wins) give each entry a normalised funder (None keeps the cleaned name) and a group.
EU_RI = "EU research programmes"
EU_SF = "EU structural & recovery funds"
GR_PUB = "Greek public funders"
OTHER_PUB = "Other public funders"
UNIV = "Universities & research centres"
INDUSTRY = "Industry"
FOUND = "Foundations & non-profits"
UNCLASS = "Unclassified"
GROUPS = [EU_RI, EU_SF, GR_PUB, OTHER_PUB, UNIV, INDUSTRY, FOUND, UNCLASS]
GROUP_COLORS = dp.color_map(GROUPS, fixed={UNCLASS: dp.OTHER_COLOR})

EC = "European Commission (programme not named)"
H2020 = "Horizon 2020"
HEU = "Horizon Europe"
FP7 = "FP7 (Seventh Framework Programme)"
MSCA = "Marie Skłodowska-Curie Actions (MSCA)"
ESF = "European Social Fund (ESF)"
ERDF = "European Regional Development Fund (ERDF)"
NSRF = "EU–Greek co-financed programmes (NSRF/ESPA)"
NGEU = "NextGenerationEU (recovery plans)"
HFRI = "Hellenic Foundation for Research and Innovation (HFRI)"
GSRT = "General Secretariat for Research and Technology (GSRT)"
IKY = "State Scholarships Foundation (IKY)"
NSFC = "National Natural Science Foundation of China (NSFC)"
EPSRC = "Engineering and Physical Sciences Research Council (EPSRC, UK)"
NSF_US = "National Science Foundation (NSF, US)"
QNRF = "Qatar National Research Fund (QNRF)"
FCT = "Fundação para a Ciência e a Tecnologia (FCT, Portugal)"
KHALIFA = "Khalifa University"
AUTH_OWN = "Aristotle University of Thessaloniki (own funds)"

FUNDER_RULES = [
    # not funders: licence or template text, place names, data providers, fragments
    (r"creative commons|widget corporation|^thessaloniki greece$|^u\.k\.$|sign language teachers|"
     r"soil data centre|ethics board|kontinuierlichen|^foundation$|european central bank|^channel$|"
     r"^assistive devices$", None, UNCLASS),
    # names that look like something else; Scopus mis-maps the acronym "EU" to Xi'an Eurasia University
    (r"horizon therapeutics", None, INDUSTRY),
    (r"xi'an eurasia university", EC, EU_RI),
    (r"european science foundation|european space agency|european metropolis", None, OTHER_PUB),
    (r"european league against rheumatism", None, FOUND),
    (r"pharmaceutical industr", None, INDUSTRY),
    (r"engineering and physical sciences", EPSRC, OTHER_PUB),
    # EU structural, cohesion and recovery funds; the Greek NSRF/ESPA programmes they co-finance.
    # Scopus maps "ESF" to a SUNY college, ERDF's Spanish acronym FEDER to a rare-disease federation
    # and the "CREATE" of the Greek RESEARCH - CREATE - INNOVATE action to the Australian "Create NSW".
    (r"european social fund|social fund plus|^esf$|\beu-esf\b|college of environmental science and forestry",
     ESF, EU_SF),
    (r"regional development fund|^erdf$|^feder$|federacion espanola de enfermedades raras", ERDF, EU_SF),
    (r"interreg|cross-border cooperation|\beni cb|balkan-?med", "Interreg & cross-border cooperation", EU_SF),
    (r"next ?generation ?eu|recovery and resilience|ryve and resilience|ripresa e resilienza|^nrrp$|^prtr$|"
     r"ecosistemi dell", NGEU, EU_SF),
    (r"competitividade|^compete 2020$", None, EU_SF),
    (r"competitiveness,? entrepreneurship|epanek|epeaek|strategic re(?:ference|search) framework|^nsrf$|"
     r"\bespa\b|european union and gree|national and community funds|^(?:greek|hellenic) (?:state|government|"
     r"republic|national funds)$|^national (?:funds|resources)|strengthening human resources research potential|"
     r"aquaculture.{0,3}industrial materials|^research$|^create$|^create nsw$|^research create$|"
     r"ministry of national education and eu$|human resources development,? education and lifelong learning",
     NSRF, EU_SF),
    # EU research and innovation programmes
    (r"marie|sk\S{0,2}odowska|\bmsca|\brise\b|\bitn\b", MSCA, EU_RI),
    (r"european research council", "European Research Council (ERC)", EU_RI),
    (r"european cooperation in science and technology|^cost$|horizon 2020 cost", "COST Actions", EU_RI),
    (r"euratom|eurofusion", "Euratom / EUROfusion", EU_RI),
    (r"horizon ?europe|horizon-ju|smart networks and services|^sns[- ]?ju$", HEU, EU_RI),
    (r"horizon ?2020|\bh2020\b|fet-?open|future (?:and )?emerging tech", H2020, EU_RI),
    (r"seventh framework|\bfp7\b|fp7[-/]", FP7, EU_RI),
    (r"sixth framework|6th framework|\bfp6\b|fp6-|research framework programme|\bfp5\b|-fp5|brite|esprit|"
     r"\brace\b|\bist\b|ten-ibc|network of excellence|^satnex$", "FP6 and earlier framework programmes", EU_RI),
    (r"innovative medicines initiative|eit health|\bempir\b|metrology (?:programme|research)|"
     r"national metrology institutes|era-?net|european joint programme|life programme|^life\+$|"
     r"^environment and climate action$|^intas$|joint undertaking|digital europe programme|"
     r"clean energy transition partnership|(?:health and digital|european research|climate, infrastructure and "
     r"environment|innovation council and smes|education and culture|^environment) executive agency", None, EU_RI),
    (r"european (?:union|commission|community|communities|committee)|^eu\b|^e\.u\.|\beu$|eu-funded|\bec$|"
     r"\bcec\b", EC, EU_RI),
    # Greek public funders
    (r"hellenic foundation for research and (?:innovation|invn)|h\.?\s?f\.?\s?r\.?\s?i\b|\bhfri\b|elidek|"
     r"horizontal support of all sciences", HFRI, GR_PUB),
    # (also a misspelt "Geneal Secretary of Research and Innovation", the secretariat's current name)
    (r"general secretariat|secretary general (?:of|for) research|\bgene?r?al secretary (?:of|for) research|"
     r"secretariat (?:for|of) (?:research|science)|\bgsrt\b|\bpened\b|reinforcement program of human research|"
     r"\byper\b|greece-montenegro", GSRT, GR_PUB),
    (r"state scholarship\S*\s*foundation|scholarships state institute|\biky\b|i\.k\.y|"
     r"national (?:fellowship|scholarships) foundation of greece", IKY, GR_PUB),
    (r"religious affairs|minist\w*.{0,70}(?:greece|greek|hellenic)|(?:greek|hellenic) minist|"
     r"ministry of national education", "Greek ministries", GR_PUB),
    (r"heal-link|hellenic academic libraries", "Hellenic Academic Libraries Link (HEAL-Link)", GR_PUB),
    (r"region of central macedonia|prefecture of crete|hellas grid|^gaec$|hellenic technology clusters|"
     r"corallia|municipal enterprise for water supply|\bgrnet\b", None, GR_PUB),
    # universities and research centres named explicitly
    (r"aristotle|a\.u\.th|^it auth$", AUTH_OWN, UNIV),
    (r"khalifa|^ku-\d|ku-kaist|kaist-ku|^ku-heig$|^kk-jrc$|healthcare engineering (?:and )?innovation cent|^heic$|"
     r"center for cyber physical|center for catalysis and separations", KHALIFA, UNIV),
    (r"foundation (?:of|for) research and technology|research and technology hellas|^certh$",
     "Greek research centres (CERTH, FORTH)", UNIV),
    # industry named explicitly (state-owned utilities included)
    (r"public power corporation|^dei$", "Public Power Corporation (PPC)", INDUSTRY),
    (r"telecommunications organization|^ote$", "OTE (Hellenic Telecommunications Organization)", INDUSTRY),
    (r"admie|independent power transmission", "IPTO (ADMIE)", INDUSTRY),
    (r"dubai electricity and water", None, INDUSTRY),
    (r"siemens foundation|vodafone foundation|pfizer foundation|merck institute", None, FOUND),
    (r"hellenic petroleum|siemens|nokia|ericsson|huawei|\bintel\b|microsoft|google|\bibm\b|"
     r"international business machines|nvidia|samsung|\babb\b|pfizer|bayer|biogen|janssen|bristol-myers|"
     r"monsanto|kpmg|mathworks|^eni$|sidenor|net2grid|qamcom|cortec|brainsway|estabanell|bell labs|"
     r"milan flug|mando group|infineon|ansys|ainigma", None, INDUSTRY),
    # foundations named explicitly
    (r"latsis", "Latsis Foundation", FOUND),
    (r"onassis", "Onassis Foundation", FOUND),
    (r"niarchos", "Stavros Niarchos Foundation", FOUND),
    (r"education and european culture|^european culture$|^foundation for education$",
     "Foundation for Education and European Culture (IPEP)", FOUND),
    (r"bodossaki", "Bodossaki Foundation", FOUND),
    (r"brain and behavior|harvard foundation|brain canada|mental health foundation|"
     r"science foundation for technology and economics|weston brain|for ?women in science|"
     r"royal academy of engineering|raeng|^royal society$|qatar foundation|hellenic society|folke hermansen|"
     r"international astronautical federation", None, FOUND),
    # other public funders named explicitly
    (r"(?<!provincial )natur\w* science (?:foundation|fund) of china|^nsfc|china nsf|nsf of china|"
     r"national science fund for distinguished young", NSFC, OTHER_PUB),
    (r"^(?:u\.?s\.? )?national science foundation$|^nsf-|directorate for mathematical|division of materials",
     NSF_US, OTHER_PUB),
    (r"^epsrc", EPSRC, OTHER_PUB),
    (r"qatar national research fund|national priorities research program", QNRF, OTHER_PUB),
    (r"fundacao para a ciencia|^foundation for science and technology$", FCT, OTHER_PUB),
    (r"national key (?:r&d|research and development) program|state.?s key project of research and development",
     "National Key R&D Program of China", OTHER_PUB),
    (r"national institutes? of (?:health|biomedical|general medical)|^nih\b",
     "National Institutes of Health (NIH, US)", OTHER_PUB),
    (r"^tekes$|finnish funding agency for technology", "Tekes (Finland)", OTHER_PUB),
    (r"academy of finland|suomen akatemia", "Academy of Finland", OTHER_PUB),
    (r"fundamental research funds for the central universities|higher education discipline innovation|"
     r"collaborative innovation fund|centro para el desarrollo tecnol|technological development centre|"
     r"centres of excellence|alexander von humboldt|canada foundation for innovation|"
     r"promotion of science|binational science|institucio catalana|canadian institutes of health|"
     r"air force research lab|china association for science|^mctes$|research grants council|"
     r"u\.?k\.? research and innovation|department (?:of|for) science,? innovation and technology|\bbmftr\b|"
     r"aperfei.{0,2}oamento de pessoal|vlaamse regering|^tamkeen$|staatssekretariat|desenvolvimento das ciencias|"
     r"turkiye bilimsel|business finland|^helse vest$|^mcin/aei|forskningsradet", None, OTHER_PUB),
    # ministries outside Greece (some names contain "Universidad" or "Università")
    (r"^minist", None, OTHER_PUB),
    # generic: universities and research centres
    (r"universi|univeristy|college|polytechnic|institute of technology|educational insti|graduate school|"
     r"instituto de investigaci|faculdade|"
     r"kaist|advanced institute of science|uc3m|\bkth\b|\bbupt\b|tekniska|\becole\b|hochschule|school of|"
     r"faculty of|department of (?!energy|education|commerce|defense|agriculture|science and technology|"
     r"the economy)|laborator|research cent|research institute|institute for systems|"
     r"centre national de la recherche|consiglio nazionale delle ricerche|\bcnr\b|inria|nazionale di fisica|"
     r"helmholtz|zentrum|academy of sciences|akademie v|praemium academiae|chinese academy|max planck|"
     r"fraunhofer|centre for|center for|cent(?:er|re)$|clinic|hospital|medical cent|medisch|renatech|"
     r"\bntu\b|ntu-|mrc-cbu|research facility|drug research academy|institute of neurology|"
     r"department of medicine|commissariat", None, UNIV),
    # generic: companies
    (r"\b(?:inc|ltd|llc|plc|gmbh|ag|bv|b\.v|ab|oy|s\.?a|corp|corporation|company)\b", None, INDUSTRY),
    # generic: Greek public bodies
    (r"greek|greece|hellenic", None, GR_PUB),
    # generic: public research foundations and funds
    (r"(?:science|research|scientific|scholar|postdoctoral|youth)\s+(?:foundation|fund)|"
     r"foundation (?:for|of) (?:science|research)|research (?:promotion|and innovation) foundation|"
     r"nationalfonds|national science fund|strategisk forskning|strategic research|foundation of shenzhen",
     None, OTHER_PUB),
    # generic: foundations, trusts, charities, societies
    (r"foundation|fondation|fundaci|stiftung|stiftelse|fonden|saatio|rahasto|\btrust\b|charit|endowment|"
     r"society|association|gesellschaft|league|\bieee\b|wellcome|leverhulme|fellowship|public benefit",
     None, FOUND),
    # generic: other public funders (national, regional and international)
    (r"\bminist|bundesminist|department (?:of|for)|government|gobierno|federal|national|nacional|council|"
     r"\bagen[cz]|agentura|commission|\bstate\b|province|provincial|municipal|\bcity\b|region|county|junta|"
     r"comunidad|generalitat|program|plan\b|office|administration|directorate|authority|academy|\bnato\b|"
     r"north atlantic treaty|unesco|atomic energy|central bank|defen[cs]e|army|naval|air force|darpa|nasa|"
     r"vinnova|vetenskapsr|formas|research fund|\bfund\b|grantov|science and technology|basic energy sciences|"
     r"high energy physics|china|chinese|beijing|guangdong|guangzhou|sichuan|jiangsu|jiangxi|zhejiang|"
     r"shandong|shanxi|shaanxi|shenzhen|hubei|fujian|liaoning|qingdao|guangxi|shanghai|canada|canadian|"
     r"ontario|\bnsf\b|\bnrf\b|\bonr\b|\baro\b|afosr|afrl|ukri|uk research and innovation|research council|"
     r"global environment facility|daad|austauschdienst|akatemia|amparo|araucaria|mokslo taryba|"
     r"forschungsgemeinschaft|\bmsit\b|^msip$|equipex|^cper$|contrat de plan|agence|research board|"
     r"scientific employment stimulus|national park|bohemian forest|stint|"
     r"medical research council|economic and social research", None, OTHER_PUB),
]
_FUNDER_RX = [(re.compile(pattern), name, group) for pattern, name, group in FUNDER_RULES]


def fold(text: object) -> str:
    """Matching key: accents removed, lower case, unified apostrophes and spaces ('' if missing)."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKD", text.replace("’", "'"))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", text).strip().lower()


def classify_funder(name: object) -> tuple[str, str]:
    """(normalised funder, funder group) of one Scopus funding entry."""
    if not isinstance(name, str) or not name.strip():
        return "(no funder name)", UNCLASS
    clean = re.sub(r"\s+", " ", name.replace("’", "'")).strip()
    key = fold(clean)
    for rx, norm, group in _FUNDER_RX:
        if rx.search(key):
            return norm or clean, group
    return clean, UNCLASS


funders = funders.join(pd.DataFrame(funders["funder"].map(classify_funder).tolist(),
                                    columns=["funder_norm", "group"], index=funders.index))
# Horizon 2020 began in 2014: earlier entries with that name are FP7 calls (Future and Emerging Technologies)
funders.loc[funders["funder_norm"].eq(H2020) & funders["year"].lt(2014), "funder_norm"] = FP7
# Scopus expands the acronym ESF to the European Science Foundation: from 2015 on, these entries stand for the
# European Social Fund or, with a COST grant, for a COST action (checked against the funding texts)
ESCI_FROM = 2015
esci = funders["funder"].eq("European Science Foundation") & funders["year"].ge(ESCI_FROM)
esci_cost = esci & funders["grants"].fillna("").str.contains(r"\bCOST\b")
funders.loc[esci_cost, ["funder_norm", "group"]] = ["COST Actions", EU_RI]
funders.loc[esci & ~esci_cost, ["funder_norm", "group"]] = [ESF, EU_SF]
n_esci = funders.loc[esci, "eid"].nunique()
# One "Create NSW" entry is the CREATE PERSWADE programme of Canada's NSERC (funding text), not the Greek action
nserc_create = funders["eid"].eq("2-s2.0-85077472632") & funders["funder"].eq("Create NSW")
funders.loc[nserc_create, ["funder_norm", "group"]] = ["Natural Sciences and Engineering Research Council of Canada",
                                                       OTHER_PUB]
# Some entries name a body abroad, or a ministry without its country, where the other funders, grant numbers or
# affiliations of the publication show another funder (Scopus files some funders under the name of another body).
# These entries are moved to the funder meant.
FP_EARLY = "FP6 and earlier framework programmes"
SERC_SE = "Swedish E-Science Research Center"
countries_of = (data.load_publication_affiliations().dropna(subset=["country_code"])
                .groupby("eid")["country_code"].agg(frozenset))
entry_countries = funders["eid"].map(lambda eid: countries_of.get(eid, frozenset()))


def no_affiliation_in(country: str) -> pd.Series:
    """Entries of publications without an affiliation in this country."""
    return ~entry_countries.map(lambda cc: country in cc)


def listed_with(funder_norm: str) -> pd.Series:
    """Entries of publications that also list this (normalised) funder."""
    return funders["eid"].isin(funders.loc[funders["funder_norm"] == funder_norm, "eid"])


STANDS_FOR = [   # (Scopus name, entries where it stands for another funder, funder meant, group)
    # Spain's Ministerio de Fomento ("Ministry of Development"): the Greek Ministry of Development, next to GSRT
    # (grant 339 is a PENED-94 grant of GSRT) or as the only funder of publications with no affiliation in Spain
    ("Ministerio de Fomento", no_affiliation_in("ES"), "Greek ministries", GR_PUB),
    # "SSF" next to IKY: the State Scholarships Foundation, not the Swedish Foundation for Strategic Research
    ("Stiftelsen för Strategisk Forskning", listed_with(IKY) & no_affiliation_in("SE"), IKY, GR_PUB),
    # Cyprus's Research and Innovation Foundation next to HFRI entries: HFRI
    ("Research and Innovation Foundation", listed_with(HFRI) & no_affiliation_in("CY"), HFRI, GR_PUB),
    # the operational programme "Human Resources Development, Education and Lifelong Learning" (with ESF and IKY)
    # and NSRF 2014-2020 (grant "2014-2020", with NSRF, ESF and IKY)
    ("Human Resources and Skills Development Canada", no_affiliation_in("CA"), NSRF, EU_SF),
    ("Institute for National Strategic Studies", no_affiliation_in("US"), NSRF, EU_SF),
    # "Network of Excellence": the FP6 network of excellence listed on the same publications (SatNEx)
    ("AUTO21 Network of Centres of Excellence", listed_with(FP_EARLY) & no_affiliation_in("CA"), FP_EARLY, EU_RI),
    # an ACTS project of FP4 (grant AC110, next to TEN-IBC), not NICE, which was founded in 1999
    ("National Institute for Health and Care Excellence",
     funders["grants"].fillna("").str.fullmatch(r"AC\d{3}") & no_affiliation_in("GB"), FP_EARLY, EU_RI),
    # a proposal number of the EU's submission system (SEP-...) next to a Horizon 2020 grant
    ("National Landcare Programme", funders["grants"].fillna("").str.startswith("SEP-") & no_affiliation_in("AU"),
     H2020, EU_RI),
    # "SeRC": the Swedish e-Science Research Centre, also listed on the publication
    ("Connecticut State Emergency Response Commission", listed_with(SERC_SE) & no_affiliation_in("US"), SERC_SE,
     UNIV),
    # a ministry that names no country, on publications with affiliations in Greece only (next to the ESF)
    ("Ministry of Education", entry_countries.map(lambda cc: cc <= {config.HOME_COUNTRY}), "Greek ministries",
     GR_PUB),
]
stands_for = pd.Series(False, index=funders.index)
for scopus_name, where, meant, group in STANDS_FOR:
    rows = funders["funder"].eq(scopus_name) & where
    funders.loc[rows, ["funder_norm", "group"]] = [meant, group]
    stands_for |= rows
n_stands_for, n_stands_for_pubs = int(stands_for.sum()), funders.loc[stands_for, "eid"].nunique()
stands_for_pubs = funders[stands_for].groupby("funder")["eid"].nunique()   # publications per Scopus name
STANDS_FOR_EXAMPLES = {"fomento": "Ministerio de Fomento", "ssf": "Stiftelsen för Strategisk Forskning",
                       "auto21": "AUTO21 Network of Centres of Excellence"}   # named in the notes
stands_for_examples = {key: int(stands_for_pubs.get(name, 0)) for key, name in STANDS_FOR_EXAMPLES.items()}
ssf_iky_entries = int((stands_for & funders["funder"].eq(STANDS_FOR_EXAMPLES["ssf"])).sum())   # entries
# The IEEE paper template thanks "the U.S. Department of Commerce under Grant BS123456": left in by the authors,
# not a funder
template = (funders["eid"].isin(funders.loc[funders["grants"].fillna("").str.contains(r"\bBS123456\b"), "eid"])
            & funders["funder"].eq("U.S. Department of Commerce"))
funders.loc[template, "group"] = UNCLASS
# Entries that cannot be attributed to any funder are left unclassified. UK Research and Innovation was founded in
# 2018: an earlier entry with its name is not UKRI (the 2003 one gives as its grant number the postcode of a company
# in the publication's affiliations). A ministry that names no country, as the only funder of a publication with
# affiliations in Greece only, may be Greek or not.
UKRI_FOUNDED = 2018
greece_only = entry_countries.map(lambda cc: cc <= {config.HOME_COUNTRY})
only_funder = funders.groupby("eid")["funder"].transform("nunique").eq(1)   # the only funder name of its publication
UNATTRIBUTED = {
    "ukri": (funders["funder"].map(fold).str.contains(r"u\.?k\.? research and innovation|\bukri\b")
             & funders["year"].lt(UKRI_FOUNDED)),
    "ministry": funders["funder"].eq("Ministry of Science Research and Technology") & greece_only & only_funder,
}
unattributed = pd.concat(UNATTRIBUTED, axis=1).any(axis=1)
funders.loc[unattributed, "group"] = UNCLASS


def on_publications(rows: pd.Series) -> str:
    """The publications of these entries, in words: 'a 2003 publication', '2 publications of 1996–2003'."""
    years = funders.loc[rows].groupby("eid")["year"].first()
    if len(years) == 1:
        return f"a {years.iloc[0]} publication"
    span = str(years.min()) if years.min() == years.max() else f"{years.min()}–{years.max()}"
    return f"{dp.fmt_int(len(years))} publications of {span}"


unattributed_pubs = {key: on_publications(rows) for key, rows in UNATTRIBUTED.items() if rows.any()}

# --------------------------------------------------------------------------- funder groups per publication
# Scopus often lists the "European Commission" next to the programme or fund it manages. Such an entry
# belongs to the EU structural funds when the publication names one of them (and no research programme),
# otherwise to the EU research programmes.
named = funders[funders["funder_norm"] != EC]
groups = (pd.crosstab(named["eid"], named["group"]).gt(0)
          .reindex(index=pubs["eid"], columns=GROUPS, fill_value=False).set_axis(pubs.index))
ec_listed = pubs["eid"].isin(funders.loc[funders["funder_norm"] == EC, "eid"])
groups[EU_RI] = groups[EU_RI] | (ec_listed & ~groups[EU_SF])
groups[UNCLASS] = groups[UNCLASS] & ~groups[GROUPS[:-1]].any(axis=1)   # only unclassified funders
pubs = pubs.join(groups)
pubs["eu_any"] = pubs[EU_RI] | pubs[EU_SF]
pubs["eu_or_greek"] = pubs["eu_any"] | pubs[GR_PUB]

# --------------------------------------------------------------------------- publisher groups
# Publisher groups with the rules of the Venues & publishers page (Publisher field, then DOI prefix,
# then venue name). Used for the latest period only, when the Publisher field is almost always filled.
PUBLISHER_RULES = [
    ("Wiley", r"wiley-ieee"),
    ("IEEE", r"\bieee\b|institute of electrical and electronics engineers"),
    ("IET", r"institution of engineering and technology|^iee$|institution of electrical engineers|"
            r"peter peregrinus"),
    ("Elsevier", r"elsevier|academic press|pergamon|north[- ]holland"),
    ("Springer Nature", r"springer|kluwer academic|nature research|nature publishing|biomed central|"
                        r"birkhauser"),
    ("MDPI", r"\bmdpi\b|multidisciplinary digital publishing"),
    ("Wiley", r"wiley|blackwell|hindawi"),
    ("Taylor & Francis", r"taylor (?:and|&) francis|routledge|crc press"),
    ("AIP Publishing", r"american institute of physics"),
    ("IOP Publishing", r"institute of physics|\biop\b"),
    ("ACM", r"association for computing machinery"),
    ("American Physical Society", r"american physical society"),
    ("Optica", r"optical society|\bosa\b|optica publishing"),
    ("SPIE", r"^spie$"),
    ("Frontiers", r"^frontiers"),
    ("SciTePress", r"scitepress|science and technology publications, lda"),
    ("Oxford University Press", r"oxford univ"),
    ("Cambridge University Press", r"cambridge univ"),
    ("World Scientific", r"^world scientific publishing|^world scientific$"),
    ("Emerald", r"^emerald"),
    ("IOS Press", r"^ios press"),
    ("SAGE", r"\bsage\b"),
    ("De Gruyter", r"de gruyter"),
    ("Wolters Kluwer", r"lippincott"),
    ("VDE Verlag", r"^vde verlag"),
]
PUBLISHER_RX = [(name, re.compile(rx)) for name, rx in PUBLISHER_RULES]
DOI_PREFIX_GROUP = {
    "10.1109": "IEEE", "10.23919": "IEEE", "10.1049": "IET",
    "10.1016": "Elsevier", "10.1006": "Elsevier",
    "10.1007": "Springer Nature", "10.1023": "Springer Nature", "10.1038": "Springer Nature",
    "10.1186": "Springer Nature", "10.1140": "Springer Nature", "10.2200": "Springer Nature",
    "10.3390": "MDPI",
    "10.1002": "Wiley", "10.1111": "Wiley", "10.1155": "Wiley", "10.1100": "Wiley",
    "10.1080": "Taylor & Francis", "10.1201": "Taylor & Francis", "10.3109": "Taylor & Francis",
    "10.1179": "Taylor & Francis",
    "10.1063": "AIP Publishing", "10.1088": "IOP Publishing", "10.1145": "ACM",
    "10.1103": "American Physical Society", "10.1364": "Optica", "10.1117": "SPIE",
    "10.3389": "Frontiers", "10.5220": "SciTePress", "10.1093": "Oxford University Press",
    "10.1017": "Cambridge University Press", "10.1142": "World Scientific", "10.1108": "Emerald",
    "10.3233": "IOS Press", "10.1177": "SAGE", "10.1515": "De Gruyter", "10.2478": "De Gruyter",
    "10.1097": "Wolters Kluwer", "10.2528": "Electromagnetics Academy",
    "10.1371": "Public Library of Science", "10.1051": "EDP Sciences",
    "10.1121": "Acoustical Society of America", "10.1115": "ASME", "10.1162": "MIT Press",
    "10.1163": "Brill", "10.1556": "Akadémiai Kiadó", "10.2316": "ACTA Press", "10.4018": "IGI Global",
    "10.1504": "Inderscience", "10.3182": "IFAC", "10.2196": "JMIR Publications", "10.1055": "Thieme",
    "10.1137": "SIAM", "10.5755": "Kaunas University of Technology", "10.3144": "BME-PT",
}
NAME_RULES = [
    ("IEEE", r"\bieee\b"),
    ("IET", r"^iee\b"),
    ("Springer Nature", r"^(?:lecture notes in|communications in computer and information science|"
                        r"ifip advances|ifmbe proceedings|mechanisms and machine science)"),
    ("CEUR-WS", r"^ceur workshop proceedings"),
    ("Optica", r"^optics infobase"),
    ("SPIE", r"^proceedings of spie"),
    ("IOS Press", r"^studies in health technology and informatics"),
    ("ACM", r"^acm international conference proceeding series"),
]
NAME_RX = [(name, re.compile(rx)) for name, rx in NAME_RULES]


def publisher_group(name: object) -> str | None:
    """Publisher group of a raw Scopus Publisher value (the raw name for small publishers)."""
    if not isinstance(name, str) or not name.strip():
        return None
    low = fold(name)
    return next((group for group, rx in PUBLISHER_RX if rx.search(low)), name.strip())


def name_group(*names: object) -> str | None:
    """Publisher implied by a source title or conference name, if any."""
    for name in names:
        low = fold(name)
        for group, rx in NAME_RX:
            if low and rx.search(low):
                return group
    return None


pub_group = pubs["publisher"].map(publisher_group).astype(object)
pub_group[pubs["source_title"].str.match(r"^IET\b", na=False) & pub_group.eq("Wiley")] = "IET"  # IET at Wiley
doi_prefix = pubs["doi"].str.extract(r"^\s*(10\.\d{4,9})/", expand=False)
pub_group = pub_group.fillna(doi_prefix.map(DOI_PREFIX_GROUP))
by_name = [name_group(s, c) for s, c in zip(pubs["source_title"], pubs["conference_name"])]
pub_group = pub_group.fillna(pd.Series(by_name, index=pubs.index, dtype=object))
pubs["publisher_group"] = pub_group.fillna("Unknown")
S_EU, S_OTHER, S_NONE = "EU research programme", "Other funding only", "No funding acknowledgement"
pubs["fund_status"] = np.select([pubs[EU_RI], pubs["has_funding"]], [S_EU, S_OTHER], default=S_NONE)

full = pubs[pubs["year"] < PARTIAL]            # full years only: shares and comparisons
recent = full[full["period"] == P_RECENT]
by_period = full.groupby("period")

# --------------------------------------------------------------------------- funding acknowledgements
n_pubs, n_funded, n_entries = len(pubs), int(pubs["has_funding"].sum()), len(funders)
funded_p = by_period["has_funding"].mean()
kind_funded = (full[full["source_kind"].isin(["journal", "conference"])]
               .groupby(["source_kind", "period"])["has_funding"].mean())
jour_funded, conf_funded = kind_funded["journal"], kind_funded["conference"]
year_stats = full.groupby("year").agg(n=("eid", "size"), funded=("has_funding", "mean"),
                                      eu_or_greek=("eu_or_greek", "mean"), other_pub=(OTHER_PUB, "mean"))
MIN_PER_YEAR = 50                              # charts start with the first period with >= 50 records every year
n_per_year = year_stats["n"].reindex(range(FIRST_YEAR, PARTIAL), fill_value=0)
START = next(first for first, _ in map(periods.bounds, periods.periods(FIRST_YEAR))
             if (n_per_year.loc[first:] >= MIN_PER_YEAR).all())
P_BASE = periods.period_of(START, first_year=FIRST_YEAR)   # the first period of the charts
EARLY = [p for p in periods.periods(FIRST_YEAR) if START <= periods.bounds(p)[0] < SPLIT]   # START to SPLIT - 1
early_text = and_join(EARLY)
early_funded_text = and_join(dp.fmt_pct(funded_p[p]) for p in EARLY)
conf_pre_max, jour_pre_min, jour_pre_max = (conf_funded[EARLY].max(), jour_funded[EARLY].min(),
                                            jour_funded[EARLY].max())
before = pubs[pubs["year"] < START]
n_before, n_funded_before = len(before), int(before["has_funding"].sum())
first_funded = int(pubs.loc[pubs["has_funding"], "year"].min())
funded_early = full.loc[full["year"].between(START, 2007), "has_funding"].mean()      # before 2008 ...
funded_late = full.loc[full["year"].between(2008, SPLIT - 1), "has_funding"].mean()   # ... and after, to SPLIT - 1
JUMP = int(year_stats.loc[START:, "funded"].diff().idxmax())   # year with the largest one-year rise
# the step in both source types (journal publications and conference papers)
kind_year = full.groupby(["source_kind", "year"])["has_funding"].mean()
jump_j0, jump_j1 = kind_year[("journal", JUMP - 1)], kind_year[("journal", JUMP)]
jump_c0, jump_c1 = kind_year[("conference", JUMP - 1)], kind_year[("conference", JUMP)]
# RESEARCH - CREATE - INNOVATE: Scopus name fragments or the action's project codes T1EDK/T2EDK (Latin or Greek
# letters, often under "Operational Program Competitiveness, Entrepreneurship and Innovation")
rci_rows = ((funders["funder"].fillna("").str.contains(r"^(?:research|create|create nsw|research create)$",
                                                       case=False, regex=True)
             | funders["grants"].fillna("").str.contains(r"[TΤ][12]\s*[EΕ][DΔ][KΚ]", regex=True))
            & ~nserc_create)
n_rci = funders.loc[rci_rows, "eid"].nunique()
rci_first, rci_last = (int(v) for v in funders.loc[rci_rows, "year"].agg(["min", "max"]))
hfri_first = int(funders.loc[funders["funder_norm"] == HFRI, "year"].min())
# the new Greek funders of the step, in words (the checks keep both within a year of JUMP)
if hfri_first == rci_first:
    new_funders_text = ("New Greek funders appear "
                        + ("in the same year" if hfri_first == JUMP else "around the same time")
                        + ": the funding agency HFRI and the RESEARCH – CREATE – INNOVATE action are both first "
                        + f"acknowledged in {hfri_first}.")
else:
    new_funders_text = ("New Greek funders appear around the same time: the funding agency HFRI is first acknowledged "
                        f"in {hfri_first} and the RESEARCH – CREATE – INNOVATE action in {rci_first}.")

# --------------------------------------------------------------------------- funders
N_TOP = 20
eu_named = named.loc[named["group"].isin([EU_RI, EU_SF]), "eid"]
counted = funders[(funders["group"] != UNCLASS)                # EC merged into the EU funder named alongside
                  & ~((funders["funder_norm"] == EC) & funders["eid"].isin(eu_named))]
fstats = (counted.groupby(["funder_norm", "group"])
          .agg(papers=("eid", "nunique"), first=("year", "min"), last=("year", "max")).reset_index())
recent_per_funder = (counted[counted["year"].between(RECENT_FROM, LAST_FULL)]
                     .groupby("funder_norm")["eid"].nunique())
fstats["papers_recent"] = fstats["funder_norm"].map(recent_per_funder).fillna(0).astype(int)
fstats["years"] = np.where(fstats["first"] == fstats["last"], fstats["first"].astype(str),
                           fstats["first"].astype(str) + "–" + fstats["last"].astype(str))
top_funders = fstats.sort_values(["papers", "funder_norm"], ascending=[False, True]).head(N_TOP)
papers_of = fstats.set_index("funder_norm")["papers"]
recent_of = fstats.set_index("funder_norm")["papers_recent"]
period_of_eid = pubs.set_index("eid")["period"]
gsrt_p = (counted[counted["funder_norm"] == GSRT].assign(period=lambda d: d["eid"].map(period_of_eid))
          .groupby("period")["eid"].nunique().reindex(periods.periods(FIRST_YEAR), fill_value=0))
P_GSRT_PEAK = gsrt_p.idxmax()                  # the period in which GSRT was acknowledged most often
gr_recent = fstats[fstats["group"] == GR_PUB].set_index("funder_norm")["papers_recent"]
# The funders after the first one, named as in the text (default: the normalised name)
TEXT_NAMES = {NSRF: "the Greek programmes co-financed by EU structural funds (NSRF/ESPA, {n})",
              ESF: "the European Social Fund ({n})", ERDF: "the European Regional Development Fund ({n})",
              NSFC: "the National Natural Science Foundation of China (NSFC, {n})", HFRI: "HFRI ({n})",
              GSRT: "GSRT ({n})", H2020: "Horizon 2020 ({n})", HEU: "Horizon Europe ({n})", FP7: "FP7 ({n})",
              MSCA: "the Marie Skłodowska-Curie Actions ({n})", KHALIFA: "Khalifa University ({n})",
              EC: "the European Commission with no programme named ({n})"}
top_next_text = and_join(TEXT_NAMES.get(name, name + " ({n})").format(n=dp.fmt_int(papers_of[name]))
                         for name in top_funders["funder_norm"].iloc[1:5])
n_top_eu = int(top_funders["group"].isin([EU_RI, EU_SF]).sum())
n_top_gr = int((top_funders["group"] == GR_PUB).sum())
n_ec_merged = int((ec_listed & pubs["eid"].isin(eu_named)).sum())
# Funders from abroad and international papers (counts over the funder totals), on both bases: address-based
# (an affiliation abroad) and external-partner (counting only affiliations of co-authors who are not SECE
# authors, so that an SECE author's own second affiliation abroad does not count)
ABROAD = {NSFC: "NSFC", KHALIFA: "Khalifa University", EPSRC: "EPSRC"}   # short names for the text
partner_eids = funders.loc[funders["funder_norm"].isin(list(ABROAD)), "eid"].unique()
partner_pubs = pubs[pubs["eid"].isin(partner_eids)]
n_partner = len(partner_pubs)
n_partner_intl = int(partner_pubs["is_international"].sum())
partner_intl_text = "all" if n_partner_intl == n_partner else dp.fmt_int(n_partner_intl)
# publications international only through such second affiliations, and which of the funders they acknowledge
abroad_sece_only = partner_pubs.loc[partner_pubs["is_international"] & ~partner_pubs["is_international_external"],
                                    "eid"]
abroad_sece_only_funders = [short for name, short in ABROAD.items()
                            if funders.loc[funders["eid"].isin(abroad_sece_only), "funder_norm"].eq(name).any()]
# In words rather than as a count: for named funders, such a count could rest on the second affiliations
# of one or two SECE authors (the site's small-group rule, config.MIN_SECE_HOLDERS).
if abroad_sece_only.empty:
    partner_ext_text = ("Counting only affiliations of co-authors who are not SECE authors, so that an SECE author's "
                        "own second affiliation abroad does not count, gives the same count.")
else:
    assert len(abroad_sece_only) <= 0.05 * n_partner, "text: almost all of them stay international"
    partner_ext_text = ("Counting only affiliations of co-authors who are not SECE authors, so that an SECE author's "
                        "own second affiliation abroad does not count, almost all of them are still international; "
                        f"the few others, which acknowledge {' or '.join(abroad_sece_only_funders)}, have no "
                        "affiliation abroad other than such second affiliations.")
n_names = funders["funder"].nunique()
n_no_name = int(funders["funder"].isna().sum())
unclass_share = (funders["group"] == UNCLASS).mean()
n_unclass_only = int(pubs[UNCLASS].sum())
mismap = {key: funders.loc[funders["funder"].fillna("").str.contains(rx, case=False, regex=True) & ~nserc_create,
                          "eid"].nunique()
          for key, rx in {"create_nsw": r"^create nsw$", "suny": r"college of environmental science and forestry",
                          "feder": r"enfermedades raras", "xian": r"xi'an eurasia university"}.items()}

# --------------------------------------------------------------------------- funding mix (funded publications)
funded_pubs = full[full["has_funding"] & (full["year"] >= START)]
mix = funded_pubs.groupby("period")[GROUPS + ["eu_any"]].mean()
mix_k = funded_pubs.groupby("period")[GROUPS].sum()
mix_n = funded_pubs.groupby("period").size()
P_GR_PEAK = mix[GR_PUB].idxmax()               # the period in which Greek public funders had their largest share
P_GR_LOW = mix[GR_PUB].idxmin()                # ... and their smallest
P_OTHER_PEAK = mix[OTHER_PUB].idxmax()         # largest share of the other public funders
gr_ratio = mix_k.at[P_RECENT, GR_PUB] / mix_k.at[P_GR_PEAK, GR_PUB]   # publications acknowledging Greek public funders
share_range = mix[GROUPS[:-1]].max() - mix[GROUPS[:-1]].min()        # how much the share of each group changed
# Other public funders: those of other countries (EU member states included) and international bodies; the notes
# name examples (checked below)
other_pub_entries = funders[funders["group"] == OTHER_PUB]
other_pub_raw = other_pub_entries["funder"].map(fold)
# international papers among the publications acknowledging other public funders (a share, so full years only),
# on both bases
other_pub_full = full[full[OTHER_PUB]]
other_pub_intl = other_pub_full["is_international"].mean()
other_pub_intl_ext = other_pub_full["is_international_external"].mean()
# Publications international only through SECE authors' own second affiliations abroad: in words, never as a number
# (next to the address-based share, a share on the external-partner basis would give away how many there are, which
# can rest on one or two SECE authors; the site's small-group rule, config.MIN_SECE_HOLDERS)
n_other_pub_sece_only = int((other_pub_full["is_international"] & ~other_pub_full["is_international_external"]).sum())
if n_other_pub_sece_only == 0:
    other_pub_ext_note = "(also when counting only affiliations of co-authors who are not SECE authors)"
else:
    assert other_pub_intl_ext >= 0.95 * other_pub_intl, "text: almost as many on the external-partner basis"
    other_pub_ext_note = "(almost as many when counting only affiliations of co-authors who are not SECE authors)"
univ_recent = recent[recent[UNIV]]
univ_top2 = univ_recent["eid"].isin(funders.loc[funders["funder_norm"].isin([KHALIFA, AUTH_OWN]), "eid"]).mean()
sf_recent = recent[recent[EU_SF]]
sf_nsrf = sf_recent["eid"].isin(funders.loc[funders["funder_norm"] == NSRF, "eid"]).mean()

# --------------------------------------------------------------------------- EU funding
eu_p = by_period[["eu_any", EU_RI, EU_SF]].mean()
eu_year = full[full["year"] >= START].groupby("year")[["eu_any", EU_RI, EU_SF]].mean()
eu_peak, ri_peak, sf_peak = (int(eu_year[c].idxmax()) for c in ("eu_any", EU_RI, EU_SF))
sf_low = int(eu_year.loc[sf_peak:, EU_SF].idxmin())        # the lowest share of structural funds since their peak
sf_fall_text = f"fell to {dp.fmt_pct(eu_year.at[sf_low, EU_SF])} in {sf_low}"
if sf_low != LAST_FULL:
    sf_fall_text += f" ({dp.fmt_pct(eu_year.at[LAST_FULL, EU_SF])} in {LAST_FULL})"
sf_peak_pubs = full[(full["year"] == sf_peak) & full[EU_SF]]
rci_share_peak = sf_peak_pubs["eid"].isin(funders.loc[rci_rows, "eid"]).mean()

# --------------------------------------------------------------------------- open access
OA_ROUTES = {"gold": "Gold", "hybrid": "Hybrid", "bronze": "Bronze", "green": "Green"}
OA_COLORS = {label: dp.CATEGORY_COLORS["oa_status"][key] for key, label in OA_ROUTES.items()}
oa_p = by_period["oa_any"].mean()
route_p = pd.crosstab(full["period"], full["oa_status"], normalize="index")
oa_year = pubs.groupby("year")["oa_any"].agg(["mean", "sum", "size"])
oa_peak = int(oa_year.loc[START:LAST_FULL, "mean"].idxmax())
green_any = pubs.groupby("year")["oa_green"].mean()            # any repository copy (also with other routes)
green_peak = int(green_any.loc[START:LAST_FULL].idxmax())
oa_after = oa_year.loc[oa_peak + 1:LAST_FULL, "mean"]       # the full years after the peak year
oa_after_text = and_join(f"{dp.fmt_pct(share)} in {year}" for year, share in oa_after.items()) if len(oa_after) else ""
green_after_text = and_join(dp.fmt_pct(green_any[year]) for year in oa_after.index) if len(oa_after) else ""
after_years_text = and_join(oa_after.index) if len(oa_after) else ""
oa_kind = recent.groupby("source_kind")["oa_any"].mean()
conf_open = recent[(recent["source_kind"] == "conference") & recent["oa_any"]]
conf_green = (conf_open["oa_status"] == "green").mean()

ieee = recent[recent["publisher_group"] == "IEEE"]
ieee_share = len(ieee) / len(recent)
ieee_kind_oa = ieee.groupby("source_kind")["oa_any"].mean()
recent_journals = recent[recent["source_kind"] == "journal"]
N_MIN_PUB = 15                                 # publishers with at least this many journal publications
journal_pub_n = recent_journals["publisher_group"].value_counts()
journal_publishers = journal_pub_n[journal_pub_n >= N_MIN_PUB].index.tolist()
journal_pub_oa = recent_journals.groupby("publisher_group")["oa_any"].mean()
fully_oa = recent_journals[recent_journals["publisher_group"].isin(["MDPI", "Frontiers"])]   # open by design
n_fully_oa, n_fully_oa_closed = len(fully_oa), int((~fully_oa["oa_any"]).sum())
if n_fully_oa_closed:
    fully_oa_text = (f"They also miss some open papers: {dp.fmt_int(n_fully_oa_closed)} of the "
                     f"{dp.fmt_int(n_fully_oa)} journal publications of {P_RECENT} from MDPI and Frontiers, which "
                     "publish only open-access journals, are flagged as closed.")
else:
    fully_oa_text = (f"As a check, all {dp.fmt_int(n_fully_oa)} journal publications of {P_RECENT} from MDPI and "
                     "Frontiers, which publish only open-access journals, are flagged as open.")

# --------------------------------------------------------------------------- open access and funding
OA_PERIODS = [P_PREV, P_RECENT]
status_oa = (pubs[pubs["period"].isin(OA_PERIODS) & pubs["source_kind"].isin(["journal", "conference"])]
             .groupby(["source_kind", "period", "fund_status"])["oa_any"].agg(["mean", "sum", "size"]))
status_oa["lo"], status_oa["hi"] = wilson(status_oa["sum"], status_oa["size"])
oa_funded = recent.groupby("has_funding")["oa_any"].mean()
oa_kind_funded = recent.groupby(["source_kind", "has_funding"])["oa_any"].mean()
pub_funded_oa = recent_journals.groupby(["publisher_group", "has_funding"])["oa_any"].mean()
oa_rate = status_oa["mean"]

# The text states these patterns of the current data: stop if new data change them.
step = {col: year_stats.at[JUMP, col] - year_stats.at[JUMP - 1, col] for col in ("funded", "eu_or_greek", "other_pub")}
other_0, other_1 = (dp.fmt_pct(year_stats.at[year, "other_pub"]) for year in (JUMP - 1, JUMP))
if abs(step["other_pub"]) < 0.005:
    other_step_text = f"hardly changed ({other_0} and {other_1})"
else:
    other_step_text = (f"{'fell slightly' if step['other_pub'] < 0 else 'rose only slightly'}, "
                       f"from {other_0} to {other_1}")
ri_j, other_j, none_j = (oa_rate[("journal", P_RECENT, s)] for s in (S_EU, S_OTHER, S_NONE))
ri_c, other_c, none_c = (oa_rate[("conference", P_RECENT, s)] for s in (S_EU, S_OTHER, S_NONE))
OTHER_GROUPS = [GR_PUB, OTHER_PUB, UNIV, INDUSTRY, FOUND]
checks = {
    "the funded share stayed below a fifth in the early periods, then rose in each later one":
        funded_p[EARLY].max() < 0.2 and funded_p[EARLY].max() < funded_p[P_PREV] < funded_p[P_RECENT],
    "conference papers changed more than journal publications and now acknowledge funding more often":
        (conf_funded[P_RECENT] - conf_funded[P_DECADE] > jour_funded[P_RECENT] - jour_funded[P_DECADE]
         and conf_funded[P_RECENT] > jour_funded[P_RECENT]),
    "the funding step came in both source types": jump_j1 > jump_j0 and jump_c1 > jump_c0,
    "the step came from EU and Greek public funding; other public funders changed by less than two points":
        step["eu_or_greek"] > step["funded"] / 2 and abs(step["other_pub"]) < 0.02,
    "HFRI and RESEARCH - CREATE - INNOVATE first appear within a year of the step":
        abs(hfri_first - JUMP) <= 1 and abs(rci_first - JUMP) <= 1,
    "no break in coverage around 2008": START < 2008 < SPLIT and abs(funded_late - funded_early) < 0.03,
    "early on, conference papers had funding details less often than journal publications":
        conf_pre_max < jour_pre_min,
    "the EU is the most frequent source of funding":
        eu_p.at[P_RECENT, "eu_any"] > recent[OTHER_GROUPS].mean().max(),
    "Horizon 2020 heads the list of funders": top_funders["funder_norm"].iloc[0] == H2020,
    "the caption's examples of combined bars are among the top funders":
        {"Greek ministries", "FP6 and earlier framework programmes"} <= set(top_funders["funder_norm"]),
    "HFRI is the most frequently acknowledged Greek public funder of the latest period": gr_recent.idxmax() == HFRI,
    "GSRT: most publications in the period just before P_DECADE, far fewer from then on, fewer than HFRI now":
        (periods.bounds(P_GSRT_PEAK)[1] + 1 == periods.bounds(P_DECADE)[0]
         and max(gsrt_p[P_DECADE], gsrt_p[P_PREV], gsrt_p[P_RECENT]) < gsrt_p[P_GSRT_PEAK] / 2
         and gsrt_p[P_RECENT] < recent_of[HFRI]),
    "the funders from abroad come with international papers": n_partner_intl >= 0.9 * n_partner,
    "the notes' examples of other public funders are in that group (EU member states and NATO among them)":
        ({NSFC, EPSRC, NSF_US, QNRF, FCT} <= set(other_pub_entries["funder_norm"])
         and other_pub_raw.str.contains("forschungsgemeinschaft").any()
         and other_pub_raw.str.contains("north atlantic treaty").any()),
    "one 'Create NSW' entry is the CREATE programme of NSERC": int(nserc_create.sum()) == 1,
    "the notes' examples of entries filed under another body are moved (Fomento and AUTO21: all their entries), most "
    "of them to Greek or EU funders":
        (all(n > 0 for n in stands_for_examples.values())
         and all(stands_for[funders["funder"].eq(STANDS_FOR_EXAMPLES[key])].all() for key in ("fomento", "auto21"))
         and funders.loc[stands_for, "group"].isin([EU_RI, EU_SF, GR_PUB]).mean() > 0.5),
    "the notes' entries that cannot be attributed are in the data": all(rows.any() for rows in UNATTRIBUTED.values()),
    "no ministry counted as a public funder abroad is on a publication with affiliations in Greece only":
        not (funders["group"].eq(OTHER_PUB) & funders["funder"].map(fold).str.match(r"minist") & greece_only).any(),
    "the notes' example of a state-owned company abroad counts with industry":
        (funders["funder"].eq("Dubai Electricity and Water Authority").any()
         and funders.loc[funders["funder"].eq("Dubai Electricity and Water Authority"), "group"].eq(INDUSTRY).all()),
    "Greek public funders: the share changed most; peak, low and latest period in this order":
        (share_range.idxmax() == GR_PUB
         and periods.bounds(P_GR_PEAK)[0] < periods.bounds(P_GR_LOW)[0] < periods.bounds(P_RECENT)[0]),
    "more publications acknowledge Greek public funders, but a smaller share of the funded ones":
        gr_ratio > 1 and mix.at[P_RECENT, GR_PUB] < mix.at[P_GR_PEAK, GR_PUB] and mix_n[P_RECENT] > mix_n[P_GR_PEAK],
    "universities and research centres reached their largest share in the latest period":
        mix[UNIV].idxmax() == P_RECENT,
    "industry and foundations remain small": mix[[INDUSTRY, FOUND]].max().max() < 0.1,
    "EU funding: larger share in each of P_BASE, P_DECADE and P_RECENT":
        eu_p.at[P_BASE, "eu_any"] < eu_p.at[P_DECADE, "eu_any"] < eu_p.at[P_RECENT, "eu_any"],
    "research programmes and structural funds followed different paths":
        (ri_peak != sf_peak and sf_low > sf_peak
         and eu_year.at[LAST_FULL, EU_RI] > eu_year.at[sf_peak, EU_RI] > eu_year.at[LAST_FULL, EU_SF]),
    "much of the structural-fund peak came from RESEARCH - CREATE - INNOVATE": rci_share_peak >= 0.4,
    "open access grew period by period": oa_p[P_BASE] < oa_p[P_DECADE] < oa_p[P_PREV] < oa_p[P_RECENT],
    "gold is the main route, ahead of green, hybrid and bronze; hybrid rose":
        (route_p.at[P_RECENT, "gold"] > route_p.at[P_RECENT, "green"] > route_p.at[P_RECENT, "hybrid"]
         > route_p.at[P_RECENT, "bronze"] and route_p.at[P_RECENT, "hybrid"] > route_p.at[P_PREV, "hybrid"]),
    "the years after the peak year have lower shares and fewer repository copies":
        (oa_peak < LAST_FULL and green_peak <= oa_peak
         and (green_any[oa_after.index] < green_any[green_peak]).all()),
    "journal publications are open far more often than conference papers, which are open mostly as green":
        oa_kind["journal"] > 2 * oa_kind["conference"] and conf_green > 0.5,
    "IEEE: journal publications open far more often than conference papers":
        ieee_kind_oa["journal"] > 2 * ieee_kind_oa["conference"],
    "Elsevier journal publications are open less often than Springer Nature and IEEE ones":
        journal_pub_oa["Elsevier"] < min(journal_pub_oa["Springer Nature"], journal_pub_oa["IEEE"]),
    "funded publications are more often open, among journal publications too":
        oa_funded[True] > oa_funded[False] and oa_kind_funded[("journal", True)] > oa_kind_funded[("journal", False)],
    "the gap holds within IEEE and Elsevier, the two largest journal publishers":
        (journal_pub_n.index[:2].tolist() == ["IEEE", "Elsevier"]
         and all(pub_funded_oa[(p, True)] > pub_funded_oa[(p, False)] for p in ("IEEE", "Elsevier"))),
    "journal publications with an EU research programme are open most often, more than in P_PREV":
        (ri_j > max(other_j, none_j, oa_kind_funded[("journal", True)])
         and ri_j > oa_rate[("journal", P_PREV, S_EU)]),
    "conference papers: only those with an EU research programme stand out":
        ri_c - max(other_c, none_c) > 0.05 and abs(other_c - none_c) < ri_c - max(other_c, none_c),
    "the intro counts the publications with funder entries": n_funded == funders["eid"].nunique(),
}
failed = [claim for claim, ok in checks.items() if not ok]
assert not failed, f"text: {failed}"

This page looks at how much of the School's research acknowledges external funding, which funders
appear most often, how the mix of funders has changed and how much of the output is openly accessible.
It draws on two fields of the {eval}`dp.fmt_int(n_pubs)` Scopus records: the funding details that Scopus
extracts from the acknowledgements of the papers ({eval}`dp.fmt_int(n_entries)` funder entries on
{eval}`dp.fmt_int(n_funded)` publications) and the open-access flags, which describe the status of each
publication on {eval}`AS_OF_TEXT`. A publication counts as funded when Scopus lists at least one funder for
it; funder names were normalised and sorted into groups (see the notes at the end). {eval}`str(PARTIAL)` is
a partial year: its records count in the funder totals but are left out of all shares, by year and by
period, and of the comparisons.

In [ ]:
dp.kpi_html([
    dp.KPI("Publications acknowledging funding", dp.fmt_pct(funded_p[P_RECENT]),
           f"{P_RECENT}; {dp.fmt_pct(funded_p[P_DECADE])} in {P_DECADE}"),
    dp.KPI("Acknowledging EU funding", dp.fmt_pct(eu_p.at[P_RECENT, "eu_any"]),
           f"{P_RECENT}; research programmes {dp.fmt_pct(eu_p.at[P_RECENT, EU_RI])}, "
           f"structural and recovery funds {dp.fmt_pct(eu_p.at[P_RECENT, EU_SF])}"),
    dp.KPI("Open access", dp.fmt_pct(oa_p[P_RECENT]), f"{P_RECENT}; {dp.fmt_pct(oa_p[P_DECADE])} in {P_DECADE}"),
    dp.KPI("Open access, journal publications", dp.fmt_pct(oa_kind["journal"]),
           f"{P_RECENT}; conference papers {dp.fmt_pct(oa_kind['conference'])}"),
], min_width=140)

:::{important} Key findings
- {eval}`dp.fmt_pct(funded_p[P_RECENT])` of the publications of {eval}`P_RECENT` acknowledge at least one
  funder, against {eval}`dp.fmt_pct(funded_p[P_DECADE])` in {eval}`P_DECADE`. The largest one-year rise since
  {eval}`str(START)` came in {eval}`str(JUMP)`, from {eval}`dp.fmt_pct(year_stats.at[JUMP - 1, 'funded'])` to
  {eval}`dp.fmt_pct(year_stats.at[JUMP, 'funded'])`, and was concentrated in EU and Greek public funding;
  part of it may reflect more complete indexing of acknowledgements by Scopus.
- The EU is the most frequent source of funding: {eval}`dp.fmt_pct(eu_p.at[P_RECENT, 'eu_any'])` of the
  {eval}`P_RECENT` publications acknowledge it, {eval}`dp.fmt_pct(eu_p.at[P_RECENT, EU_RI])` through EU research
  programmes and {eval}`dp.fmt_pct(eu_p.at[P_RECENT, EU_SF])` through EU structural and recovery funds.
  Horizon 2020 is the most frequently acknowledged funder ({eval}`dp.fmt_int(papers_of[H2020])`
  publications).
- HFRI, first acknowledged in {eval}`str(hfri_first)`, is now the most frequently acknowledged Greek public
  funder: {eval}`dp.fmt_int(recent_of[HFRI])` publications of {eval}`P_RECENT` acknowledge it. GSRT was
  acknowledged by {eval}`dp.fmt_int(gsrt_p[P_GSRT_PEAK])` publications of {eval}`P_GSRT_PEAK`, but already by only
  {eval}`dp.fmt_int(gsrt_p[P_DECADE])` in {eval}`P_DECADE` and by {eval}`dp.fmt_int(gsrt_p[P_RECENT])` in
  {eval}`P_RECENT`.
- {eval}`dp.fmt_pct(oa_p[P_RECENT])` of the {eval}`P_RECENT` publications are open access, against
  {eval}`dp.fmt_pct(oa_p[P_DECADE])` in {eval}`P_DECADE`; gold is the main route
  ({eval}`dp.fmt_pct(route_p.at[P_RECENT, 'gold'])`). Journal publications are open far more often
  ({eval}`dp.fmt_pct(oa_kind['journal'])`) than conference papers ({eval}`dp.fmt_pct(oa_kind['conference'])`).
- Funded journal publications of {eval}`P_RECENT` are open access more often than unfunded ones
  ({eval}`dp.fmt_pct(oa_kind_funded[('journal', True)])` against
  {eval}`dp.fmt_pct(oa_kind_funded[('journal', False)])`), and those acknowledging an EU research programme
  most often ({eval}`dp.fmt_pct(oa_rate[('journal', P_RECENT, S_EU)])`).
:::

In [ ]:
#| label: cell-funding-share-year
SERIES_COLORS = {"All publications": dp.INK["muted"], "Journal publications": KIND_COLORS["journal"],
                 "Conference papers": KIND_COLORS["conference"]}
since = full[full["year"] >= START]           # shares: full years only
parts = {"All publications": since,
         "Journal publications": since[since["source_kind"] == "journal"],
         "Conference papers": since[since["source_kind"] == "conference"]}
share_year = pd.concat(
    [part.groupby("year")["has_funding"].agg(share="mean", funded="sum", n="size").reset_index().assign(series=name)
     for name, part in parts.items()],
    ignore_index=True)
MIN_N = 10                                   # shares of fewer publications are not drawn
small = share_year[share_year["n"] < MIN_N]
small_text = "; ".join(f"{s.lower()} of {y}: {n}" for s, y, n in zip(small["series"], small["year"], small["n"]))
share_year = share_year[share_year["n"] >= MIN_N]
small_note = (f" Shares of fewer than {MIN_N} publications are not drawn ({small_text} publications)."
              if len(small) else "")
fig = dp.year_line(share_year, "year", "share", color="series", colors=SERIES_COLORS,
                   value_title="Publications with a funding acknowledgement", value_format=".1%",
                   custom_data=["funded", "n"])
fig.update_traces(hovertemplate="%{y:.1%} (%{customdata[0]:,} of %{customdata[1]:,})")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

## Funding acknowledgements over time

:::{figure} #cell-funding-share-year
:label: fig-funding-share-year
Share of publications with at least one funding acknowledgement in Scopus, by year, for all publications
and by source type (numbers in the hover).{eval}`small_note` {eval}`str(PARTIAL)` (partial year) is not
shown.
:::

The share of publications with a funding acknowledgement stayed below a fifth in {eval}`early_text`
({eval}`early_funded_text`), then rose to {eval}`dp.fmt_pct(funded_p[P_PREV])` in {eval}`P_PREV` and
{eval}`dp.fmt_pct(funded_p[P_RECENT])` in {eval}`P_RECENT` ([](#fig-funding-share-year)). Conference papers
changed most: {eval}`dp.fmt_pct(conf_funded[P_DECADE])` of them acknowledged funding in {eval}`P_DECADE` and
{eval}`dp.fmt_pct(conf_funded[P_RECENT])` in {eval}`P_RECENT`, when they did so more often than journal
publications ({eval}`dp.fmt_pct(jour_funded[P_RECENT])`).

In {eval}`str(JUMP)`, the year with the largest increase since {eval}`str(START)`, the share rose in journal
publications ({eval}`dp.fmt_pct(jump_j0)` to {eval}`dp.fmt_pct(jump_j1)`) and conference papers
({eval}`dp.fmt_pct(jump_c0)` to {eval}`dp.fmt_pct(jump_c1)`) alike. A step of this size within one year
may partly reflect more complete indexing of acknowledgements by Scopus, but it is concentrated in EU and
Greek public funding: the share of publications acknowledging EU or Greek public funders rose from
{eval}`dp.fmt_pct(year_stats.at[JUMP - 1, 'eu_or_greek'])` to
{eval}`dp.fmt_pct(year_stats.at[JUMP, 'eu_or_greek'])`, while the share acknowledging other public funders
(those of other countries and international bodies) {eval}`other_step_text`.
{eval}`new_funders_text`

The chart starts in {eval}`str(START)`: from that five-year period on, the School published at least
{eval}`str(MIN_PER_YEAR)` papers every year, so that yearly shares do not rest on a handful of papers.
Scopus lists funders for {eval}`dp.fmt_int(n_funded_before)` of the {eval}`dp.fmt_int(n_before)` earlier
publications, the first from {eval}`str(first_funded)`. The data show no break in coverage around 2008:
{eval}`dp.fmt_pct(funded_early)` of the publications of {eval}`str(START)`–2007 and
{eval}`dp.fmt_pct(funded_late)` of those of 2008–{eval}`str(SPLIT - 1)` have funding details.

In [ ]:
#| label: cell-funding-top-funders
top = top_funders.sort_values(["papers", "funder_norm"], ascending=[True, False])   # plotly draws bottom-up
top = top.assign(label=[dp.wrap_label(name, 36) for name in top["funder_norm"]])
fig = go.Figure()
for grp in GROUPS:
    part = top[top["group"] == grp]
    if part.empty:
        continue
    fig.add_bar(
        x=part["papers"], y=part["label"], orientation="h", name=grp, marker_color=GROUP_COLORS[grp],
        customdata=part[["funder_norm", "papers_recent", "years"]].to_numpy(),
        hovertemplate=("<b>%{customdata[0]}</b><br>" + grp + "<br>Publications: %{x:,}<br>" + P_RECENT
                       + ": %{customdata[1]:,}<br>Years: %{customdata[2]}<extra></extra>"),
        texttemplate="%{x:,}", textposition="outside",
    )
fig.update_layout(barmode="overlay", height=dp.fig_height(len(top), per_row=32) + 30, bargap=0.3, margin=dict(t=24))
fig.update_yaxes(categoryorder="array", categoryarray=top["label"].tolist(), showgrid=False, showline=True)
fig.update_xaxes(title_text="Publications", showgrid=True, showline=False, range=[0, top["papers"].max() * 1.15])
fig.show()

## Who funds the research

:::{figure} #cell-funding-top-funders
:label: fig-funding-top-funders
The {eval}`str(N_TOP)` most frequently acknowledged funders, {eval}`str(FIRST_YEAR)`–{eval}`str(PARTIAL)`:
publications that acknowledge each funder, coloured by funder group (hover: publications of
{eval}`P_RECENT` and years with acknowledgements). Names are normalised, and a few bars combine closely
related bodies, such as the successive Greek ministries or the framework programmes before FP7. A
publication can acknowledge several funders.
:::

Horizon 2020 heads the list ({eval}`dp.fmt_int(papers_of[H2020])` publications), followed by
{eval}`top_next_text` ([](#fig-funding-top-funders)). Of the {eval}`str(N_TOP)` funders,
{eval}`str(n_top_eu)` are EU programmes or funds and {eval}`str(n_top_gr)` are Greek public bodies. The
ranking has changed over time: GSRT was acknowledged by {eval}`dp.fmt_int(gsrt_p[P_GSRT_PEAK])` publications in
{eval}`P_GSRT_PEAK`, {eval}`dp.fmt_int(gsrt_p[P_DECADE])` in {eval}`P_DECADE`, {eval}`dp.fmt_int(gsrt_p[P_PREV])` in
{eval}`P_PREV` and {eval}`dp.fmt_int(gsrt_p[P_RECENT])` in {eval}`P_RECENT`, while HFRI was acknowledged by
{eval}`dp.fmt_int(recent_of[HFRI])` publications in {eval}`P_RECENT`.

Funders from abroad come with the School's international links: {eval}`partner_intl_text` of the
{eval}`dp.fmt_int(n_partner)` publications that acknowledge the National Natural Science Foundation of China
({eval}`dp.fmt_int(papers_of[NSFC])`), Khalifa University ({eval}`dp.fmt_int(papers_of[KHALIFA])`) or the
UK's EPSRC ({eval}`dp.fmt_int(papers_of[EPSRC])`) are international papers (have an affiliation abroad).
{eval}`partner_ext_text`

In [ ]:
#| label: cell-funding-mix
ROWS = GROUPS[:-1]                                   # funder groups without "Unclassified"
mix_periods = list(mix.index)                        # P_BASE ... P_RECENT
share = mix.loc[mix_periods, ROWS].T
count = mix_k.loc[mix_periods, ROWS].T
short = {p: f"{first}–{str(last)[-2:]}" for p, (first, last) in zip(mix_periods, map(periods.bounds, mix_periods))}
fig = go.Figure(go.Heatmap(
    z=share.to_numpy(),
    x=[f"{short[p]}<br>n={dp.fmt_int(mix_n[p])}" for p in mix_periods],
    y=ROWS,
    text=[[dp.fmt_pct(v) for v in row] for row in share.to_numpy()],
    texttemplate="%{text}",
    hovertext=[[f"<b>{g}</b><br>{p}: {dp.fmt_pct(share.at[g, p])} of the funded publications<br>"
                f"({dp.fmt_int(count.at[g, p])} of {dp.fmt_int(mix_n[p])})" for p in mix_periods] for g in ROWS],
    hovertemplate="%{hovertext}<extra></extra>",
    zmin=0, zmax=float(np.ceil(share.to_numpy().max() * 10) / 10),
    colorscale=dp.SEQUENTIAL_SCALE, showscale=False, xgap=2, ygap=2,
))
fig.update_xaxes(showgrid=False, showline=False, automargin=True)
fig.update_yaxes(showgrid=False, showline=False, autorange="reversed", automargin=True)
fig.update_layout(height=400, margin=dict(t=8, b=70))
fig.show()

## Funding mix over time

:::{figure} #cell-funding-mix
:label: fig-funding-mix
Share of the funded publications that acknowledge at least one funder in each group, by five-year
period, {eval}`str(START)`–{eval}`str(LAST_FULL)` (n = funded publications). A publication can count in
several groups, so the columns do not add up to 100%. {eval}`str(PARTIAL)` is not shown (partial year).
:::

EU research programmes are acknowledged by {eval}`dp.fmt_pct(mix.at[P_RECENT, EU_RI])` of the funded
publications of {eval}`P_RECENT` and EU structural and recovery funds by
{eval}`dp.fmt_pct(mix.at[P_RECENT, EU_SF])`; together, {eval}`dp.fmt_pct(mix.at[P_RECENT, 'eu_any'])` of the
funded publications acknowledge EU funding ([](#fig-funding-mix)). Of the {eval}`P_RECENT` publications with
EU structural or recovery funds, {eval}`dp.fmt_pct(sf_nsrf)` name one of the Greek NSRF/ESPA programmes that
these funds co-finance.

The share of Greek public funders changed most. They appeared on
{eval}`dp.fmt_pct(mix.at[P_GR_PEAK, GR_PUB])` of the funded publications of {eval}`P_GR_PEAK`
({eval}`dp.fmt_int(mix_k.at[P_GR_PEAK, GR_PUB])` publications), on {eval}`dp.fmt_pct(mix.at[P_GR_LOW, GR_PUB])` in
{eval}`P_GR_LOW` ({eval}`dp.fmt_int(mix_k.at[P_GR_LOW, GR_PUB])`) and on {eval}`dp.fmt_pct(mix.at[P_RECENT, GR_PUB])`
in {eval}`P_RECENT` ({eval}`dp.fmt_int(mix_k.at[P_RECENT, GR_PUB])`). Their number grew
{eval}`dp.fmt_num(gr_ratio, 1)` times from {eval}`P_GR_PEAK` to {eval}`P_RECENT`; the share fell because the
funded publications multiplied ({eval}`dp.fmt_int(mix_n[P_GR_PEAK])` to {eval}`dp.fmt_int(mix_n[P_RECENT])`).
Other public funders (those of other countries, EU member states included, and international bodies) peaked
at {eval}`dp.fmt_pct(mix.at[P_OTHER_PEAK, OTHER_PUB])` in {eval}`P_OTHER_PEAK`;
{eval}`dp.fmt_pct(other_pub_intl)` of the publications acknowledging them in
{eval}`str(FIRST_YEAR)`–{eval}`str(LAST_FULL)` are international papers {eval}`other_pub_ext_note`.
Universities and research centres reached {eval}`dp.fmt_pct(mix.at[P_RECENT, UNIV])` in {eval}`P_RECENT`
({eval}`dp.fmt_pct(univ_top2)` of these publications acknowledge Khalifa University or AUTH's own funds),
while industry ({eval}`dp.fmt_pct(mix.at[P_RECENT, INDUSTRY])`) and foundations
({eval}`dp.fmt_pct(mix.at[P_RECENT, FOUND])`) remain small.

In [ ]:
#| label: cell-funding-eu
EU_SERIES = {"eu_any": "Any EU funding", EU_RI: EU_RI, EU_SF: EU_SF}
EU_COLORS = {"Any EU funding": dp.INK["muted"], EU_RI: GROUP_COLORS[EU_RI], EU_SF: GROUP_COLORS[EU_SF]}
since = full[full["year"] >= START]           # shares: full years only
n_year = since.groupby("year").size()
eu_long = pd.concat(
    [since.groupby("year")[col].agg(share="mean", count="sum").reset_index()
     .assign(series=label, n=lambda d: d["year"].map(n_year)) for col, label in EU_SERIES.items()],
    ignore_index=True)
fig = dp.year_line(eu_long, "year", "share", color="series", colors=EU_COLORS,
                   value_title="Share of publications", value_format=".1%", custom_data=["count", "n"])
fig.update_traces(hovertemplate="%{y:.1%} (%{customdata[0]:,} of %{customdata[1]:,})")
fig.update_yaxes(tickformat=".0%", rangemode="tozero")
fig.show()

## EU funding

:::{figure} #cell-funding-eu
:label: fig-funding-eu
Share of publications acknowledging EU funding, by year from {eval}`str(START)`: EU research programmes (the
framework programmes
up to Horizon Europe, with the Marie Skłodowska-Curie Actions, the ERC and COST), EU structural and recovery
funds (ESF, ERDF, the NSRF/ESPA programmes co-financed by Greece, Interreg, NextGenerationEU) and either of
the two (numbers in the hover). {eval}`str(PARTIAL)` (partial year) is not shown.
:::

In {eval}`P_RECENT`, {eval}`dp.fmt_pct(eu_p.at[P_RECENT, 'eu_any'])` of the publications acknowledged EU
funding, against {eval}`dp.fmt_pct(eu_p.at[P_DECADE, 'eu_any'])` in {eval}`P_DECADE` and
{eval}`dp.fmt_pct(eu_p.at[P_BASE, 'eu_any'])` in {eval}`P_BASE`; the highest yearly share was
{eval}`dp.fmt_pct(eu_year.at[eu_peak, 'eu_any'])` in {eval}`str(eu_peak)` ([](#fig-funding-eu)). The two
kinds of EU funding followed different paths. Research programmes reached their highest share in
{eval}`str(ri_peak)` ({eval}`dp.fmt_pct(eu_year.at[ri_peak, EU_RI])`). Structural and recovery funds peaked
at {eval}`dp.fmt_pct(eu_year.at[sf_peak, EU_SF])` in {eval}`str(sf_peak)` and {eval}`sf_fall_text`. Much of
the peak came from the
Greek RESEARCH – CREATE – INNOVATE action: it is acknowledged on {eval}`dp.fmt_pct(rci_share_peak)` of the
{eval}`str(sf_peak)` publications with EU structural funds, and on {eval}`dp.fmt_int(n_rci)` publications in
all, from {eval}`str(rci_first)` to {eval}`str(rci_last)`.

In [ ]:
#| label: cell-funding-oa-year
since = full[full["year"] >= START]           # shares: full years only
oa_long = (pd.crosstab(since["year"], since["oa_status"], normalize="index")
           .reindex(columns=list(OA_ROUTES), fill_value=0.0).rename(columns=OA_ROUTES)
           .reset_index().melt(id_vars="year", var_name="route", value_name="share"))
fig = dp.year_bar(oa_long, "year", "share", color="route", colors=OA_COLORS,
                  value_title="Share of publications", value_format=".1%")
total = oa_year.loc[START:LAST_FULL]
fig.add_scatter(x=total.index, y=total["mean"], mode="markers", marker=dict(opacity=0, color=dp.OTHER_COLOR),
                name="Open access, all routes", showlegend=False, customdata=total[["sum", "size"]].to_numpy(),
                hovertemplate="%{y:.1%} (%{customdata[0]:,} of %{customdata[1]:,})")
fig.update_yaxes(tickformat=".0%", range=[0, float(total["mean"].max()) * 1.1])
fig.show()

## Open access over time

:::{figure} #cell-funding-oa-year
:label: fig-funding-oa-year
Share of publications that are open access, by route and year from {eval}`str(START)`, as flagged by Scopus
on {eval}`AS_OF_TEXT`.
Gold: published in a fully open-access journal; hybrid: open article in a subscription journal; bronze:
free to read on the publisher's site without an open licence; green: only a repository copy is open. A
publication with several routes counts once, in the first route of this list. {eval}`str(PARTIAL)`
(partial year) is not shown.
:::

Open access grew from {eval}`dp.fmt_pct(oa_p[P_BASE])` of the publications of {eval}`P_BASE` and
{eval}`dp.fmt_pct(oa_p[P_DECADE])` of those of {eval}`P_DECADE` to {eval}`dp.fmt_pct(oa_p[P_PREV])` in
{eval}`P_PREV` and {eval}`dp.fmt_pct(oa_p[P_RECENT])` in {eval}`P_RECENT`; the highest yearly share was
{eval}`dp.fmt_pct(oa_year.at[oa_peak, 'mean'])` in {eval}`str(oa_peak)` ([](#fig-funding-oa-year)). Gold is now
the main route ({eval}`dp.fmt_pct(route_p.at[P_RECENT, 'gold'])` of the {eval}`P_RECENT` publications), ahead
of green ({eval}`dp.fmt_pct(route_p.at[P_RECENT, 'green'])`), hybrid
({eval}`dp.fmt_pct(route_p.at[P_RECENT, 'hybrid'])`, up from {eval}`dp.fmt_pct(route_p.at[P_PREV, 'hybrid'])` in
{eval}`P_PREV`) and bronze ({eval}`dp.fmt_pct(route_p.at[P_RECENT, 'bronze'])`).

The lower shares of the years after {eval}`str(oa_peak)` ({eval}`oa_after_text`) are at least partly a
matter of timing: a repository copy is flagged for {eval}`dp.fmt_pct(green_any[green_peak])` of the
publications of {eval}`str(green_peak)` but for {eval}`green_after_text` of those of {eval}`after_years_text`.
Such copies often appear only after a publisher's embargo, so the shares of the most
recent years are likely to rise in later snapshots.

In [ ]:
#| label: cell-funding-oa-publishers
KIND_LABELS = {"journal": "Journal publications", "conference": "Conference papers", "book": "Books and chapters"}


def oa_by(df: pd.DataFrame, key: str, keys: list, labels: dict | None = None) -> pd.DataFrame:
    """Open share by route and in total for the given values of `key` (first key drawn on top)."""
    sub = df[df[key].isin(keys)]
    tab = (pd.crosstab(sub[key], sub["oa_status"], normalize="index")
           .reindex(index=keys, columns=list(OA_ROUTES), fill_value=0.0))
    tab["oa"] = sub.groupby(key)["oa_any"].mean()
    tab["n"] = sub.groupby(key).size()
    tab["label"] = [f"{(labels or {}).get(k, k)} (n = {dp.fmt_int(n)})" for k, n in zip(tab.index, tab["n"])]
    return tab.iloc[::-1]


panels = [oa_by(recent, "source_kind", list(KIND_LABELS), KIND_LABELS),
          oa_by(recent_journals, "publisher_group", journal_publishers)]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.16,
                    row_heights=[len(tab) for tab in panels],
                    subplot_titles=["By source type", "Journal publications by publisher"])
for row, tab in enumerate(panels, start=1):
    for key, route in OA_ROUTES.items():
        fig.add_bar(x=tab[key], y=tab["label"], orientation="h", name=route, legendgroup=route,
                    showlegend=row == 1, marker_color=OA_COLORS[route], hovertemplate="%{x:.1%}", row=row, col=1)
    for label, total_share in zip(tab["label"], tab["oa"]):
        fig.add_annotation(x=total_share, y=label, text=dp.fmt_pct(total_share), xanchor="left", xshift=4,
                           showarrow=False, row=row, col=1)
for title in fig.layout.annotations[:2]:             # the subplot titles, left-aligned
    title.update(x=0, xanchor="left", font=dict(size=13))
fig.update_layout(barmode="stack", hovermode="y unified", bargap=0.3,
                  height=dp.fig_height(sum(len(tab) for tab in panels)) + 130, margin=dict(t=72),
                  legend=dict(yref="container", y=1, yanchor="top"))
fig.update_xaxes(tickformat=".0%", range=[0, 1.12], tickvals=[0, 0.25, 0.5, 0.75, 1], showgrid=True,
                 showline=False)
fig.update_xaxes(title_text="Share of publications", row=2, col=1)
fig.update_yaxes(showgrid=False, showline=True)
fig.show()

## Open access by source type and publisher

:::{figure} #cell-funding-oa-publishers
:label: fig-funding-oa-publishers
Open-access share by route for the publications of {eval}`P_RECENT`: by source type (top) and, for
journal publications, by publisher (bottom: publishers with at least {eval}`str(N_MIN_PUB)` journal
publications in these years). The number at the end of each bar is the share open by any route.
Publisher imprints are grouped as on the [Venues & publishers](venues.ipynb) page.
:::

Journal publications are open far more often ({eval}`dp.fmt_pct(oa_kind['journal'])` in {eval}`P_RECENT`) than
conference papers ({eval}`dp.fmt_pct(oa_kind['conference'])`), which are open mostly through repository
copies: {eval}`dp.fmt_pct(conf_green)` of the open conference papers are green open access
([](#fig-funding-oa-publishers)). The difference shows within IEEE, which published
{eval}`dp.fmt_pct(ieee_share)` of the {eval}`P_RECENT` output: {eval}`dp.fmt_pct(ieee_kind_oa['journal'])` of its
journal publications are open, but only {eval}`dp.fmt_pct(ieee_kind_oa['conference'])` of its conference
papers. Among journal publishers, MDPI ({eval}`dp.fmt_pct(journal_pub_oa['MDPI'])`) and Frontiers
({eval}`dp.fmt_pct(journal_pub_oa['Frontiers'])`) publish only open-access journals;
{eval}`dp.fmt_pct(journal_pub_oa['Springer Nature'])` of the Springer Nature and
{eval}`dp.fmt_pct(journal_pub_oa['IEEE'])` of the IEEE journal publications are open, but only
{eval}`dp.fmt_pct(journal_pub_oa['Elsevier'])` of the Elsevier ones.

In [ ]:
#| label: cell-funding-oa-by-funding
STATUSES = [S_EU, S_OTHER, S_NONE]
STATUS_COLORS = {S_EU: GROUP_COLORS[EU_RI], S_OTHER: dp.CATEGORICAL[2], S_NONE: dp.OTHER_COLOR}
KIND_TITLES = {"journal": "Journal publications", "conference": "Conference papers"}
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=list(KIND_TITLES.values()))
for col, kind in enumerate(KIND_TITLES, start=1):
    for status in STATUSES:
        part = status_oa.loc[kind].xs(status, level="fund_status").reindex(OA_PERIODS)
        fig.add_bar(
            x=OA_PERIODS, y=part["mean"], name=status, legendgroup=status, showlegend=col == 1,
            marker_color=STATUS_COLORS[status],
            error_y=dict(type="data", symmetric=False, array=part["hi"] - part["mean"],
                         arrayminus=part["mean"] - part["lo"], color=dp.INK["muted"], thickness=1, width=3),
            customdata=part[["sum", "size"]].to_numpy(),
            hovertemplate="%{y:.1%} (%{customdata[0]:,} of %{customdata[1]:,})", row=1, col=col,
        )
fig.update_annotations(font_size=13)                # the panel titles
fig.update_layout(barmode="group", hovermode="x unified", margin=dict(t=64),
                  legend=dict(yref="container", y=1, yanchor="top"))
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.update_yaxes(title_text="Share open access", row=1, col=1)
fig.show()

## Open access and funding

:::{figure} #cell-funding-oa-by-funding
:label: fig-funding-oa-by-funding
Open-access share by funding status, source type and period: publications acknowledging an EU
research programme, publications with other funding only and publications without a funding
acknowledgement. Error bars: 95% Wilson confidence intervals (numbers of publications in the hover).
:::

Funded publications are more often open access: in {eval}`P_RECENT`, {eval}`dp.fmt_pct(oa_funded[True])` of the
funded and {eval}`dp.fmt_pct(oa_funded[False])` of the other publications were open. The gap holds among
journal publications ({eval}`dp.fmt_pct(oa_kind_funded[('journal', True)])` against
{eval}`dp.fmt_pct(oa_kind_funded[('journal', False)])`) and within their two largest publishers (IEEE
{eval}`dp.fmt_pct(pub_funded_oa[('IEEE', True)])` against {eval}`dp.fmt_pct(pub_funded_oa[('IEEE', False)])`,
Elsevier {eval}`dp.fmt_pct(pub_funded_oa[('Elsevier', True)])` against
{eval}`dp.fmt_pct(pub_funded_oa[('Elsevier', False)])`). It is largest for publications acknowledging an EU
research programme: {eval}`dp.fmt_pct(oa_rate[('journal', P_RECENT, S_EU)])` of such journal publications were
open in {eval}`P_RECENT`, up from {eval}`dp.fmt_pct(oa_rate[('journal', P_PREV, S_EU)])` in {eval}`P_PREV`
([](#fig-funding-oa-by-funding)); Horizon 2020 and Horizon Europe require open access to the peer-reviewed
publications of the projects they fund. Among conference papers only those acknowledging an EU research
programme stand out ({eval}`dp.fmt_pct(oa_rate[('conference', P_RECENT, S_EU)])` open, against
{eval}`dp.fmt_pct(oa_rate[('conference', P_RECENT, S_OTHER)])` with other funding only and
{eval}`dp.fmt_pct(oa_rate[('conference', P_RECENT, S_NONE)])` without an acknowledgement).

These are associations, not effects: funded and unfunded publications also differ in topic, venue and
collaboration, and the comparison does not control for these factors.

## Notes & caveats

- **Funding data.** Scopus extracts funder names and grant numbers from the acknowledgements of the papers.
  The data are incomplete: a missing entry does not prove that a publication was unfunded. Funding details
  are particularly sparse for older conference papers: in the five-year periods from {eval}`str(START)` to
  {eval}`str(SPLIT - 1)`, Scopus lists funders for at most {eval}`dp.fmt_pct(conf_pre_max)` of the conference
  papers, against {eval}`dp.fmt_pct(jour_pre_min)`–{eval}`dp.fmt_pct(jour_pre_max)` of the journal
  publications. Counts are publications that acknowledge a funder, not amounts of money or numbers of
  grants.
- **Normalisation.** The {eval}`dp.fmt_int(n_names)` distinct funder names were mapped by rules to normalised
  names (spelling variants, translations, programme and agency names) and to seven groups. Entries that Scopus
  files under the wrong body are moved to the funder meant when the data show which one it is. Scopus records
  the Greek RESEARCH – CREATE – INNOVATE action as the Australian
  agency "Create NSW" ({eval}`dp.fmt_int(mismap['create_nsw'])` publications) or as the fragments "RESEARCH"
  and "CREATE"; one "Create NSW" entry is in fact the CREATE programme of Canada's NSERC. The "College of
  Environmental Science and Forestry" of the State University of New York
  ({eval}`dp.fmt_int(mismap['suny'])` publications), filed under the acronym ESF, is the European Social Fund.
  The European Science Foundation ({eval}`dp.fmt_int(n_esci)` publications since {eval}`str(ESCI_FROM)`),
  also filed under ESF, is the European Social Fund or, with a COST grant, a COST action. The Spanish
  rare-disease federation FEDER ({eval}`dp.fmt_int(mismap['feder'])`) is the European Regional Development
  Fund (FEDER in Spanish), and Xi'an Eurasia University ({eval}`dp.fmt_int(mismap['xian'])`), filed under the
  acronym EU, is the European Union.
  Publications of RESEARCH – CREATE – INNOVATE are found by these names or by the action's project codes
  (T1EDK, T2EDK). Horizon 2020 entries on publications from before 2014, when the programme began, count as
  FP7. Some entries name a body abroad, or a ministry without its country, where the other funders,
  grant numbers or affiliations of the publication point to another funder, in most cases a Greek or EU one:
  these {eval}`dp.fmt_int(n_stands_for)` entries on {eval}`dp.fmt_int(n_stands_for_pubs)` publications are moved
  to that funder. Spain's "Ministerio de Fomento"
  ({eval}`dp.fmt_int(stands_for_examples['fomento'])` publications), for example, is the Greek Ministry
  of Development; the Swedish Foundation for Strategic Research stands for IKY, the State Scholarships
  Foundation (both "SSF"), in {eval}`dp.fmt_int(ssf_iky_entries)` of its entries; and Canada's AUTO21 network
  ({eval}`dp.fmt_int(stands_for_examples['auto21'])`) is the FP6 network of
  excellence listed with it. The sample acknowledgement of the IEEE paper template, to the U.S. Department of
  Commerce, is left unclassified, and so are entries whose funder cannot be identified, among them "UK Research
  and Innovation" on {eval}`unattributed_pubs['ukri']`, before UKRI was founded in 2018, and a ministry that
  names no country as the only funder of {eval}`unattributed_pubs['ministry']` with affiliations in Greece
  only. {eval}`dp.fmt_pct(unclass_share)` of the funding entries remain
  unclassified (mostly project names, fragments and {eval}`dp.fmt_int(n_no_name)` entries without a funder
  name); {eval}`dp.fmt_int(n_unclass_only)` funded publications have no classified funder.
- **Groups.** *EU research programmes*: the framework programmes up to Horizon Europe, including the Marie
  Skłodowska-Curie Actions, the ERC, COST and other programmes managed by the European Commission. *EU
  structural & recovery funds*: the European Social Fund, the European Regional Development Fund, the Greek
  NSRF/ESPA operational programmes that these funds co-finance with national funds (among them RESEARCH –
  CREATE – INNOVATE), Interreg and NextGenerationEU; the national share of the co-financing is not
  separated. *Greek public funders*: HFRI, GSRT, IKY, ministries and other public bodies. *Other public
  funders*: the national and regional public funders of other countries, EU member states included, and
  international bodies (for example NSFC, EPSRC, NSF, QNRF, Portugal's FCT, Germany's DFG and NATO).
  *Universities & research centres* include AUTH's own funds and Khalifa University; *Industry* includes the
  Public Power Corporation and OTE, and state-owned companies abroad such as the Dubai Electricity and Water
  Authority.
- **The European Commission entry.** Scopus often lists "European Commission" next to a programme or fund
  that the Commission manages. These entries are merged into the EU programme or fund named on the same
  publication ({eval}`dp.fmt_int(n_ec_merged)` publications); on the other {eval}`dp.fmt_int(papers_of[EC])`
  publications no programme is named and the entry counts with the EU research programmes.
- **Open-access flags.** Scopus takes its open-access flags from Unpaywall. They describe the status on
  {eval}`AS_OF_TEXT`, not at publication, and change as embargoes end and repositories are indexed.
  {eval}`fully_oa_text`
- **Partial year.** The {eval}`str(PARTIAL)` records (to {eval}`AS_OF_TEXT`) count in the funder totals but
  not in the shares by year or period, the funding mix or the comparisons.
- **Associations.** Differences between groups of publications (funded or not, source types, publishers)
  are descriptive; they do not show that funding or a publisher causes open access, or the reverse.
- **Small groups.** The page gives no number of papers that are international only through SECE authors' own
  second affiliations abroad, since such a number can describe one or two people; it describes these papers
  in words instead. This also covers numbers that could be worked out from the figures shown, such as the
  difference between two shares of the same publications with and without these affiliations.